# Diagnóstico de cancro da mama — Exploração dos dados

**Inteligência Artificial Centrada no Humano** · Mestrado em Inteligência Artificial e Engenharia de Dados · FCTUC

Conhecer o dataset antes de treinar: quantas imagens há por classe, como são, e quanta memória ocupam consoante o tamanho escolhido (ponto levantado pelo professor: as imagens podem ser pesadas e pode ser preciso limitar o dataset).

**Índice**
1. Configuração
2. Distribuição das classes
3. Exemplos de imagens
4. Tamanho das imagens
5. Balanceamento
6. Memória necessária
7. Conclusões

## 1. Configuração

In [ ]:
# Ambiente: monta o Google Drive no Colab; localmente usa a pasta do repositório
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/My Drive/miacd-iach-breast-cancer-diagnosis")  # pasta do repositório no Drive
    %pip install -q modAL-python scikeras
else:
    ROOT = Path.cwd().parent

sys.path.insert(0, str(ROOT))
from src.config import DATA_PATH, RESULTS_PATH

## 2. Distribuição das classes

As classes vêm do nome de cada ficheiro (ex.: `benign (12).png`).

In [ ]:
from src.exploration import class_counts, list_images, plot_class_distribution

names, labels = list_images(DATA_PATH)
counts = class_counts(labels)
print(f"Total: {len(names)} imagens", counts)
plot_class_distribution(counts, "Dataset original")

O dataset é desequilibrado: há mais do triplo de imagens benignas do que normais. Sem balanceamento, o modelo tende a favorecer a classe maioritária.

## 3. Exemplos de imagens

In [ ]:
from src.exploration import plot_examples

plot_examples(DATA_PATH, names, labels, per_class=4)

## 4. Tamanho das imagens

As imagens têm tamanhos diferentes e são todas redimensionadas para `IMG_SIZE × IMG_SIZE` no carregamento.

In [ ]:
import numpy as np
from src.exploration import original_sizes

sizes = original_sizes(DATA_PATH, names)
print(f"Largura: {sizes[:, 0].min()}–{sizes[:, 0].max()} px (mediana {int(np.median(sizes[:, 0]))})")
print(f"Altura:  {sizes[:, 1].min()}–{sizes[:, 1].max()} px (mediana {int(np.median(sizes[:, 1]))})")

## 5. Balanceamento

A divisão treino/teste (80/20, estratificada) é feita **antes** de balancear, para que versões rodadas da mesma imagem não fiquem em treino e em teste (data leakage). Só o treino é balanceado, com rotações e reflexões (`src/data.py`).

In [ ]:
from src.data import load_dataset

IMG_SIZE = 64   # pequeno, só para ver as contagens rapidamente
x_train, x_test, y_train, y_test = load_dataset(DATA_PATH, size=IMG_SIZE)
plot_class_distribution(class_counts(list(y_train)), "Treino depois do balanceamento")
plot_class_distribution(class_counts(list(y_test)), "Teste (imagens originais)")

## 6. Memória necessária

Memória só das imagens (float32, escala de cinzento), sem contar o modelo nem o treino. Ajuda a escolher `IMG_SIZE` e `MAX_PER_CLASS` nos notebooks seguintes.

In [ ]:
from src.exploration import memory_mb

n_images = len(x_train) + len(x_test)
for size in [64, 128, 224]:
    print(f"{size}×{size}: {memory_mb(n_images, size):6.0f} MB")

## 7. Conclusões

*A preencher: tamanho de imagem escolhido e se é preciso limitar o dataset.*